# HMS — learning curves, protocol v2

**Kaggle settings:** Accelerator **GPU** · **Internet on** (TabPFN's package and weights) ·
Inputs: the **HMS competition** + your **`hms-code`** dataset (latest version) · **Save Version → Save & Run All**.

Protocol v2: signals at 200 Hz (0.3–75 Hz), patient-level **and class-stratified** split, N = training
recordings, primary metric = KL on the high-quality test windows (>= 10 votes), CNN in PyTorch.
The data are prepared in `/tmp` first (about 10–15 min, not kept), so the saved output is only the results.

**To resume or add a model:** also attach the output of an earlier v2 run of this notebook (Add Input → Your Work).
Its results are copied in first; finished runs are skipped.

In [ ]:
MODELS = ['floor', 'lgbm', 'tabpfn', 'minirocket', 'cnn1d']   # files in models/
SIZES = [100, 300, 1000, 3000, 10000, 'all']           # training recordings; 'all' = the whole training pool
SEEDS = [0, 1, 2]

import os, sys, glob, json, shutil
def find(pattern):
    hits = glob.glob('/kaggle/input/' + pattern, recursive=True)
    return os.path.dirname(hits[0]) if hits else None

CODE = os.environ.get('HMS_CODE') or find('**/run.py')
if CODE is None:
    raise FileNotFoundError('run.py not found under /kaggle/input: add your hms-code dataset as an input')
sys.path.insert(0, CODE)
COMP = os.environ.get('HMS_COMP') or next((os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
                                           if os.path.isdir(os.path.join(os.path.dirname(p), 'train_eegs'))), None)
PREP = os.environ.get('HMS_PREP') or find('**/hms-prepared-v2/X.npy') or '/tmp/hms-prepared-v2'
RESULTS = os.environ.get('HMS_RESULTS', '/kaggle/working/results')
# results of an earlier v2 run attached as an input (they hold preds/hq_test.npy): copied in, not redone
PREV = os.environ.get('HMS_PREV') or next((os.path.dirname(os.path.dirname(p)) for p in
        glob.glob('/kaggle/input/**/preds/hq_test.npy', recursive=True)), None)
if PREV and os.path.abspath(PREV) != os.path.abspath(RESULTS):
    shutil.copytree(PREV, RESULTS, dirs_exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
print('code:', CODE, '\ncompetition:', COMP, '\nprepared data:', PREP, '\nresults:', RESULTS,
      '\nprevious results copied from:', PREV, '(%d runs)' % len(glob.glob(os.path.join(RESULTS, '*.json'))))

### 0. TabPFN's package (only if `tabpfn` is in MODELS)
Needs **Internet on**. Installed before anything imports scikit-learn, because this TabPFN version installs scikit-learn 1.6.

In [ ]:
if 'tabpfn' in MODELS:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabpfn==2.1.3'], check=True)

### 1. Prepare the data if none is attached
One primary window per recording plus every non-overlapping labelled window (about 49,000 windows at 200 Hz,
3.2 GB in `/tmp`). The split is printed: patients, recordings, high-quality windows and class mix per set.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import hms_data, hms_eval, run
if not os.path.exists(os.path.join(PREP, 'X.npy')):
    if COMP is None:
        raise FileNotFoundError('No prepared data and no competition data found: add the HMS competition as an input')
    hms_data.prepare(COMP, PREP)
data = hms_data.load(PREP)
display(pd.DataFrame(hms_data.describe_split(data.meta)).T)

### 2. Learning rate of the CNN, chosen on the validation set
Once, at N = 1,000, seed 0, from `models/deep.LR_GRID`: the rate with the lowest KL on the high-quality validation
windows is used for every N and seed (test windows are not used). Saved in `results/tuning.json`, so a resumed run
reuses it.

In [ ]:
TUNE = os.path.join(RESULTS, 'tuning.json')
tuned = json.load(open(TUNE)) if os.path.exists(TUNE) else {}
for m in [m for m in MODELS if m == 'cnn1d' and m not in tuned]:
    import importlib
    from models.deep import LR_GRID
    mod = importlib.import_module('models.' + m)
    sp = hms_data.make_split(data, 1000, 0)
    scores = {}
    for lr in LR_GRID:
        pv, _, info = mod.fit_predict(sp, seed=0, lr=lr)
        scores[lr] = float(hms_eval.kl_per_sample(sp.Y_val[sp.val_hq], pv[sp.val_hq]).mean())
        print('%s  lr %g: validation KL (hq) %.4f, best epoch %d of %d' % (m, lr, scores[lr], info['best_epoch'], info['epochs']))
    tuned[m] = dict(lr=min(scores, key=scores.get), val_kl_hq={str(k): round(v, 5) for k, v in scores.items()})
    json.dump(tuned, open(TUNE, 'w'), indent=1)
    del sp
print(tuned)

### 3. All the runs
Each run is its own Python process (`run.py`): libraries cannot clash, memory is freed between runs, and a failed
run is reported without stopping the others. A run whose result already exists is skipped, so a stopped session can
resume. TabPFN prints "skip" for the full pool (its limit is 10,000).

In [ ]:
import time, importlib, subprocess
names = {m: importlib.import_module('models.' + m).NAME for m in MODELS}
pool = int(((data.meta['split'] == 'train') & data.meta['is_primary']).sum())
del data                                              # each run reloads what it needs
t0, failed = time.time(), []
for n in SIZES:
    for seed in SEEDS:
        for m in MODELS:
            tag = '%s_n%d_s%d' % (names[m], pool if n == 'all' or n >= pool else n, seed)
            if os.path.exists(os.path.join(RESULTS, tag + '.json')):
                continue                                  # done (this session or an attached earlier run)
            kw = ['--kw', 'lr=%g' % tuned[m]['lr']] if m in tuned else []
            r = subprocess.run([sys.executable, os.path.join(CODE, 'run.py'), '--model', m, '--n', str(n),
                                '--seed', str(seed), '--data', PREP, '--out', RESULTS, '--skip-existing'] + kw,
                               capture_output=True, text=True)
            lines = [l for l in r.stdout.splitlines() if l.strip()]
            print(lines[-1] if lines else '(no output)', flush=True)
            if r.returncode != 0:
                failed.append(tag)
                print('FAILED %s:\n%s' % (tag, r.stderr[-2000:]), flush=True)
print('all runs done in %.0f min; failed: %s' % ((time.time() - t0) / 60, failed or 'none'))

### 4. Results: table and learning curves
Primary metric: test KL on the high-quality windows (>= 10 votes), mean over seeds (SD in brackets). Then the same on
one window per test recording ("all"), and the full summary. Figures and tables are saved in `results/figures/`.
`best_epoch_frac` close to 1 for the CNN means it was still improving at its last epoch.

In [ ]:
FIGS = os.path.join(RESULTS, 'figures')
os.makedirs(FIGS, exist_ok=True)
s = hms_eval.summarise(RESULTS)
s.to_csv(os.path.join(FIGS, 'summary.csv'), index=False)
for metric, name in (('test_kl_hq', 'high-quality windows'), ('test_kl_all', 'one window per recording')):
    t = hms_eval.curve_table(RESULTS, metric=metric)
    t.to_csv(os.path.join(FIGS, '%s_table.csv' % metric))
    print('test KL, %s' % name); display(t)
hms_eval.plot_curve(RESULTS)
plt.savefig(os.path.join(FIGS, 'learning_curves.png'), dpi=160, bbox_inches='tight'); plt.show()
hms_eval.plot_curve(RESULTS, metric='test_kl_all')
plt.savefig(os.path.join(FIGS, 'learning_curves_all_windows.png'), dpi=160, bbox_inches='tight'); plt.show()
display(s)

### 5. Paired comparisons
Same test windows (high-quality set), model A minus model B, for every N: `mean_delta` over seeds (negative = A better)
and in how many seeds each model is significantly better (bootstrap over test patients, p < 0.05).
`DETAILS = True` prints the per-seed tables.

In [ ]:
pt = hms_eval.paired_table(RESULTS)
pt.to_csv(os.path.join(FIGS, 'paired_table.csv'), index=False)
display(pt)
DETAILS = False
if DETAILS:
    for pair in pt.pair.unique():
        a, b = pair.split(' - ')
        print('\n%s minus %s' % (a, b)); display(hms_eval.paired(RESULTS, a, b))

### 6. Per class, and by number of votes
AUC of each class against the rest (majority label, high-quality windows) at N = 1,000 and 10,000; then the test KL
by number of votes, one window per test recording.

In [ ]:
hms_eval.plot_per_class_auc(RESULTS, sizes=(1000, 10000))
plt.savefig(os.path.join(FIGS, 'per_class_auc.png'), dpi=160, bbox_inches='tight'); plt.show()
for n in (1000, 10000):
    if n in set(s.n_train):
        kv = hms_eval.kl_by_votes(RESULTS, n)
        kv.to_csv(os.path.join(FIGS, 'kl_by_votes_n%d.csv' % n))
        print('test KL by number of votes, N = %d' % n); display(kv)

### 7. One file to download
`hms_results.zip`: every run, its predictions, the CNN's tuning and `figures/`.

In [ ]:
print(shutil.make_archive('/kaggle/working/hms_results', 'zip', RESULTS))